# Autonomous Executive Assistant Sandbox

Notebook for OpenRouter Gemma rollouts, checkpoint export, and RL training. Use the `scalerhack2-training` kernel so the environment matches the validated training pipeline.

## Workflow

1. Load `.env.training` directly from the repository root.
2. Run the baseline suite to confirm the environment is stable.
3. Run an OpenRouter Gemma rollout if the API key is available.
4. Export traces for analysis or imitation-style warm starts.
5. Train the tabular RL agent and save a checkpoint.
6. Promote stable changes back into `src/` and keep tests green.

In [1]:
import json
import os
from pathlib import Path

from src.executive_assistant.agent import BaselineAgent, OpenRouterPolicy
from src.executive_assistant.config import OpenRouterConfig, load_env_file
from src.executive_assistant.runner import EpisodeRunner, export_traces_jsonl, run_policy_suite
from src.executive_assistant.training import evaluate_q_policy, train_q_learning


In [2]:
ENV_FILE = Path('.env.training')
ENV_LOADED = load_env_file(ENV_FILE)
HAS_OPENROUTER_KEY = bool(os.environ.get('OPENROUTER_API_KEY'))

TASK_NAME = 'hard_rag_reply'
POLICY_PROVIDER = 'openrouter' if HAS_OPENROUTER_KEY else 'baseline'
MODEL_NAME = os.environ.get('OPENROUTER_MODEL', 'google/gemma-4-31b-it')
MAX_STEPS = 12
TRACE_DIR = Path('artifacts/traces')
CHECKPOINT_DIR = Path('artifacts/checkpoints')
TRACE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

{
    'env_file_found': ENV_LOADED,
    'has_openrouter_key': HAS_OPENROUTER_KEY,
    'policy_provider': POLICY_PROVIDER,
    'model_name': MODEL_NAME,
}


{'env_file_found': True,
 'has_openrouter_key': True,
 'policy_provider': 'openrouter',
 'model_name': 'google/gemma-4-31b-it'}

In [3]:
def build_policy(provider: str, model_name: str):
    if provider == 'baseline':
        return BaselineAgent()
    if provider == 'openrouter':
        config = OpenRouterConfig.from_env(ENV_FILE)
        config = OpenRouterConfig(
            api_key=config.api_key,
            model_name=model_name,
            base_url=config.base_url,
            site_url=config.site_url,
            app_name=config.app_name,
            temperature=config.temperature,
            max_tokens=config.max_tokens,
        )
        return OpenRouterPolicy(config=config)
    raise ValueError(f'Unsupported provider: {provider}')


## Baseline validation

Run this first. If the baseline is not still solving the seeded tasks, stop and fix the environment before trusting any LLM or RL results.

In [4]:
baseline_traces = run_policy_suite(
    policy=BaselineAgent(),
    task_names=[
        'easy_deadline_extraction',
        'medium_triage_and_negotiation',
        'hard_rag_reply',
    ],
    max_steps=MAX_STEPS,
)

{name: {'completed': trace.completed, 'score': trace.final_score, 'steps': len(trace.steps)} for name, trace in baseline_traces.items()}


{'easy_deadline_extraction': {'completed': True, 'score': 1.0, 'steps': 5},
 'medium_triage_and_negotiation': {'completed': True,
  'score': 1.0,
  'steps': 5},
 'hard_rag_reply': {'completed': True, 'score': 1.0, 'steps': 3}}

## Policy rollout

This uses OpenRouter Gemma automatically when `.env.training` provides the key. Otherwise it falls back to the baseline policy.

In [5]:
policy = build_policy(POLICY_PROVIDER, MODEL_NAME)
runner = EpisodeRunner(policy=policy, max_steps=MAX_STEPS)
trace = runner.run(TASK_NAME)

print(json.dumps(trace.to_dict(), indent=2))


{
  "task_name": "hard_rag_reply",
  "policy_name": "OpenRouterPolicy",
  "steps": [
    {
      "step_index": 1,
      "reasoning": "Following the hard_rag_reply guidance, I must read the stakeholder email first before searching for the report.",
      "action": {
        "action_type": "read_email",
        "target_id": 1,
        "payload": null,
        "secondary_payload": null
      },
      "observation": {
        "current_time": "2026-04-04T10:00:00Z",
        "unread_emails": [],
        "active_todos": [],
        "last_action_status": "email read",
        "current_email": {
          "id": 1,
          "sender": "vip.stakeholder@company.com",
          "recipient": "assistant@workspace.local",
          "subject": "Need Q3 architecture metrics",
          "body": "Please share the key Q3 architecture metrics from the report.",
          "timestamp": "2026-04-04T07:30:00Z"
        },
        "search_results": [],
        "action_history": [
          "read_email: email read

In [6]:
trace.steps[-1].snapshot


{'emails': [{'id': 1,
   'sender': 'vip.stakeholder@company.com',
   'recipient': 'assistant@workspace.local',
   'subject': 'Need Q3 architecture metrics',
   'body': 'Please share the key Q3 architecture metrics from the report.',
   'timestamp': '2026-04-04T07:30:00Z',
   'is_read': 1,
   'is_archived': 0}],
 'todos': [],
 'files': [{'id': 1,
   'filename': 'Q3_Architecture_Report.txt',
   'content_text': 'Q3 Architecture Report\nSystem availability: 99.95%\nMean API latency: 182ms\nInfrastructure cost reduction: 14%\n'}],
 'action_log': [{'id': 1,
   'action_type': 'read_email',
   'target_id': 1,
   'payload': None,
   'secondary_payload': None,
   'status': 'email read'},
  {'id': 2,
   'action_type': 'search_files',
   'target_id': None,
   'payload': 'Q3 architecture report',
   'secondary_payload': None,
   'status': '1 file(s) matched'},
  {'id': 3,
   'action_type': 'reply',
   'target_id': 1,
   'payload': 'Hello,\n\nThe key Q3 architecture metrics are: System availability:

## Export traces

These JSONL traces are the main interface between rollout collection and downstream training or regression analysis.

In [7]:
suite_traces = run_policy_suite(
    policy=build_policy(POLICY_PROVIDER, MODEL_NAME),
    task_names=[TASK_NAME],
    max_steps=MAX_STEPS,
)

output_path = export_traces_jsonl(
    list(suite_traces.values()),
    TRACE_DIR / f'{POLICY_PROVIDER}_{TASK_NAME}_traces.jsonl',
)

print(output_path)


artifacts/traces/openrouter_hard_rag_reply_traces.jsonl


## RL training

This trains the tabular Q-learning policy with a baseline-teacher warm start, saves a checkpoint, and evaluates the trained policy on all seeded tasks.

In [8]:
q_policy, training_scores = train_q_learning(
    episodes=300,
    epsilon=0.15,
    teacher=BaselineAgent(),
)
checkpoint_path = q_policy.save(CHECKPOINT_DIR / 'q_policy_notebook.json')
evaluation = evaluate_q_policy(q_policy)

{
    'checkpoint': str(checkpoint_path),
    'training_scores': training_scores,
    'evaluation': evaluation,
}


{'checkpoint': 'artifacts/checkpoints/q_policy_notebook.json',
 'training_scores': {'easy_deadline_extraction': 1.0,
  'medium_triage_and_negotiation': 0.2,
  'hard_rag_reply': 1.0},
 'evaluation': {'easy_deadline_extraction': 1.0,
  'medium_triage_and_negotiation': 1.0,
  'hard_rag_reply': 1.0}}

## Environment note

The notebook loads `.env.training` directly from the repo root. That keeps CLI runs, notebook runs, and Jupyter-launched kernels aligned without requiring manual exports in the shell.